# LeetCode #126: Word Ladder II

https://leetcode.com/problems/word-ladder-ii/

## Comparison of Approaches

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| Brute DFS (all paths) | Exponential | O(n) | TLE — explores non-shortest paths |
| **BFS layers + DFS backtrack** | **O(n·L²)** | **O(n·L)** | **Optimal — collect only shortest paths** |

## Understanding the Method

Phase 1 — BFS builds a directed acyclic graph (DAG):
- Level-by-level expansion ensures shortest paths
- Track parents: for each word, which words led to it
- Stop expanding a word once it appears at any level

Phase 2 — DFS backtrack from endWord to beginWord using the DAG, collect paths.

## Constraints

- `1 <= beginWord.length <= 5`
- `endWord.length == beginWord.length`
- `1 <= wordList.length <= 500`
- All words have the same length and consist of lowercase English letters.

## Solutions

### C#

In [ ]:
public class Solution {
    public IList<IList<string>> FindLadders(string beginWord, string endWord, IList<string> wordList) {
        var wordSet = new HashSet<string>(wordList);
        var result = new List<IList<string>>();
        if (!wordSet.Contains(endWord)) return result;
        // parent map: word -> set of predecessors
        var parents = new Dictionary<string, HashSet<string>>();
        var currLevel = new HashSet<string> { beginWord };
        bool found = false;
        while (currLevel.Count > 0 && !found) {
            wordSet.ExceptWith(currLevel); // remove visited
            var nextLevel = new HashSet<string>();
            foreach (var word in currLevel) {
                char[] arr = word.ToCharArray();
                for (int i = 0; i < arr.Length; i++) {
                    char orig = arr[i];
                    for (char c = 'a'; c <= 'z'; c++) {
                        if (c == orig) continue;
                        arr[i] = c;
                        string next = new string(arr);
                        if (wordSet.Contains(next)) {
                            nextLevel.Add(next);
                            if (!parents.ContainsKey(next)) parents[next] = new HashSet<string>();
                            parents[next].Add(word);
                            if (next == endWord) found = true;
                        }
                        arr[i] = orig;
                    }
                }
            }
            currLevel = nextLevel;
        }
        if (!found) return result;
        var path = new List<string> { endWord };
        void Dfs(string word) {
            if (word == beginWord) { result.Add(new List<string>(Enumerable.Reverse(path))); return; }
            if (!parents.ContainsKey(word)) return;
            foreach (var prev in parents[word]) { path.Add(prev); Dfs(prev); path.RemoveAt(path.Count - 1); }
        }
        Dfs(endWord);
        return result;
    }
}

### Python

In [ ]:
from collections import defaultdict, deque

class Solution:
    def findLadders(self, beginWord: str, endWord: str, wordList: list[str]) -> list[list[str]]:
        word_set = set(wordList)
        if endWord not in word_set:
            return []
        parents = defaultdict(set)
        curr_level = {beginWord}
        found = False
        while curr_level and not found:
            word_set -= curr_level
            next_level = set()
            for word in curr_level:
                for i in range(len(word)):
                    for c in 'abcdefghijklmnopqrstuvwxyz':
                        nw = word[:i] + c + word[i+1:]
                        if nw in word_set:
                            next_level.add(nw)
                            parents[nw].add(word)
                            if nw == endWord:
                                found = True
            curr_level = next_level
        if not found:
            return []
        result = []
        def dfs(word, path):
            if word == beginWord:
                result.append(path[::-1])
                return
            for p in parents[word]:
                dfs(p, path + [p])
        dfs(endWord, [endWord])
        return result

### Go

In [ ]:
func findLadders(beginWord string, endWord string, wordList []string) [][]string {
    wordSet := map[string]bool{}
    for _, w := range wordList { wordSet[w] = true }
    result := [][]string{}
    if !wordSet[endWord] { return result }
    parents := map[string][]string{}
    currLevel := map[string]bool{beginWord: true}
    found := false
    for len(currLevel) > 0 && !found {
        for w := range currLevel { delete(wordSet, w) }
        nextLevel := map[string]bool{}
        for word := range currLevel {
            bs := []byte(word)
            for i := 0; i < len(bs); i++ {
                orig := bs[i]
                for c := byte('a'); c <= 'z'; c++ {
                    if c == orig { continue }
                    bs[i] = c
                    nw := string(bs)
                    if wordSet[nw] {
                        nextLevel[nw] = true
                        parents[nw] = append(parents[nw], word)
                        if nw == endWord { found = true }
                    }
                    bs[i] = orig
                }
            }
        }
        currLevel = nextLevel
    }
    if !found { return result }
    var dfs func(string, []string)
    dfs = func(word string, path []string) {
        if word == beginWord {
            cp := make([]string, len(path))
            for i, s := range path { cp[len(path)-1-i] = s }
            result = append(result, cp)
            return
        }
        for _, p := range parents[word] { dfs(p, append(path, p)) }
    }
    dfs(endWord, []string{endWord})
    return result
}

### Rust

In [ ]:
use std::collections::{HashMap, HashSet};

impl Solution {
    pub fn find_ladders(begin_word: String, end_word: String, word_list: Vec<String>) -> Vec<Vec<String>> {
        let mut word_set: HashSet<String> = word_list.into_iter().collect();
        let mut result = vec![];
        if !word_set.contains(&end_word) { return result; }
        let mut parents: HashMap<String, Vec<String>> = HashMap::new();
        let mut curr_level: HashSet<String> = [begin_word.clone()].into();
        let mut found = false;
        while !curr_level.is_empty() && !found {
            for w in &curr_level { word_set.remove(w); }
            let mut next_level = HashSet::new();
            for word in &curr_level {
                let bs: Vec<u8> = word.bytes().collect();
                for i in 0..bs.len() {
                    for c in b'a'..=b'z' {
                        if c == bs[i] { continue; }
                        let mut nb = bs.clone();
                        nb[i] = c;
                        let nw = String::from_utf8(nb).unwrap();
                        if word_set.contains(&nw) {
                            next_level.insert(nw.clone());
                            parents.entry(nw.clone()).or_default().push(word.clone());
                            if nw == end_word { found = true; }
                        }
                    }
                }
            }
            curr_level = next_level;
        }
        if !found { return result; }
        fn dfs(word: &str, begin: &str, parents: &HashMap<String, Vec<String>>, path: &mut Vec<String>, result: &mut Vec<Vec<String>>) {
            if word == begin {
                let mut p = path.clone(); p.reverse(); result.push(p);
                return;
            }
            if let Some(preds) = parents.get(word) {
                for p in preds { path.push(p.clone()); dfs(p, begin, parents, path, result); path.pop(); }
            }
        }
        let mut path = vec![end_word.clone()];
        dfs(&end_word, &begin_word, &parents, &mut path, &mut result);
        result
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `beginWord="hit"`, `endWord="cog"`, `wordList=["hot","dot","dog","lot","log","cog"]`  
Two shortest paths of length 5.  
**Output:** `[["hit","hot","dot","dog","cog"],["hit","hot","lot","log","cog"]]`

### 2. Slightly Complex
**Input:** `beginWord="hit"`, `endWord="cog"`, wordList without "cog"  
endWord not reachable.  
**Output:** `[]`

### 3. Edge Case: Time Factor
**Input:** Large word list, many near-equal words  
BFS limits exploration to shortest level only — no extra depth.  
**Output:** All shortest paths.

### 4. Edge Case: Space Factor
**Input:** 500 words of length 5  
Parent map stores at most O(n·L) data.  
**Output:** All shortest transformation sequences.

### 5. Almost-Impossible but Plausible
**Input:** `beginWord == endWord`  
No transformation needed (but endWord must be in wordList).  
**Output:** `[[beginWord]]`

![image.png](attachment:image.png)